# Step 3: Baseline Characteristics Analysis

**Pipeline step:** Compare baseline demographics and clinical characteristics between treatment groups before outcome analysis.

**Study:** Observational comparative effectiveness — Metformin users vs Other diabetes medication users

**Input:** `reports/study_population.csv` (Step 2 cohort)

**Outputs:**
- `reports/baseline_characteristics_table.csv` — Table 1
- `reports/baseline_characteristics_report.html` — full report
- `reports/figures/` — age distribution, treatment sizes, Love plot (SMD)

## Statistical methods

| Variable type | Summary statistics | Primary test | Balance metric |
| --- | --- | --- | --- |
| Continuous (age) | Mean (SD), median (IQR), min/max | Welch t-test or Mann-Whitney U* | SMD |
| Categorical (gender, race) | n (%) | Chi-square | SMD per level |
| Binary comorbidities | n (%) | Chi-square / Fisher exact | SMD |

\* Mann-Whitney U is used when Shapiro-Wilk rejects normality in either treatment group.

**SMD interpretation:** |SMD| < 0.1 indicates good baseline balance (pharmaceutical RWE convention).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "synthea_csv"
REPORT_DIR = PROJECT_ROOT / "reports"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from baseline_analysis import run_baseline_analysis, TREATMENT_A, TREATMENT_B

In [ ]:
result = run_baseline_analysis(DATA_DIR, REPORT_DIR)

patient_data = result.patient_data
table1 = result.table1

print(f"Cohort N = {len(patient_data)}")
print(f"Treatment A ({TREATMENT_A}): {(patient_data['treatment_group'] == TREATMENT_A).sum()}")
print(f"Treatment B ({TREATMENT_B}): {(patient_data['treatment_group'] == TREATMENT_B).sum()}")

## Table 1 — Baseline characteristics

Columns show Treatment A (Metformin users), Treatment B (Other diabetes medication users), and Overall cohort statistics with p-values and standardized mean differences.

In [ ]:
display_cols = [
    "section", "characteristic", TREATMENT_A, TREATMENT_B, "overall",
    "smd", "p_value", "significance", "primary_test", "balanced",
]
display(table1[display_cols])

## Visualizations

In [ ]:
from IPython.display import Image, display

for fig_path in result.figure_paths:
    print(fig_path.name)
    display(Image(filename=str(fig_path)))

## Interpretation

Review variables with **|SMD| ≥ 0.1** or **p < 0.05** — these indicate baseline imbalance and potential confounding by indication. In the next pipeline step (propensity score estimation), these covariates should be included in the propensity model to reduce confounding before treatment effect estimation.

Open the full HTML report at `reports/baseline_characteristics_report.html` for study documentation.